In [13]:
import numpy as np
import pandas as pd

In [2]:
def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return Z > 0


def softmax(Z):
    exp = np.exp(Z - np.max(Z, axis=1, keepdims=True))
    return exp / np.sum(exp, axis=1, keepdims=True)


In [3]:
def init_params():
    W1 = np.random.randn(784, 256) * np.sqrt(2./784)
    b1 = np.zeros((1,256))

    W2 = np.random.randn(256,128) * np.sqrt(2./256)
    b2 = np.zeros((1,128))

    W3 = np.random.randn(128,26) * np.sqrt(2./128)
    b3 = np.zeros((1,26))

    return W1,b1,W2,b2,W3,b3


In [4]:
def forward(X, params):

    W1,b1,W2,b2,W3,b3 = params

    Z1 = X @ W1 + b1
    A1 = relu(Z1)

    Z2 = A1 @ W2 + b2
    A2 = relu(Z2)

    Z3 = A2 @ W3 + b3
    A3 = softmax(Z3)

    cache = (Z1,A1,Z2,A2,Z3,A3)

    return A3, cache


In [5]:
def compute_loss(y_pred, y_true):

    m = y_true.shape[0]

    log_likelihood = -np.log(y_pred[range(m), y_true] + 1e-9)

    return np.sum(log_likelihood) / m


In [6]:
 def backward(X, y, params, cache):

    W1,b1,W2,b2,W3,b3 = params
    Z1,A1,Z2,A2,Z3,A3 = cache

    m = X.shape[0]

    dZ3 = A3
    dZ3[range(m), y] -= 1
    dZ3 /= m

    dW3 = A2.T @ dZ3
    db3 = np.sum(dZ3, axis=0, keepdims=True)

    dA2 = dZ3 @ W3.T
    dZ2 = dA2 * relu_derivative(Z2)

    dW2 = A1.T @ dZ2
    db2 = np.sum(dZ2, axis=0, keepdims=True)

    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * relu_derivative(Z1)

    dW1 = X.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)

    grads = (dW1,db1,dW2,db2,dW3,db3)

    return grads


In [7]:
def update(params, grads, lr=0.001):

    updated = []

    for p,g in zip(params, grads):
        updated.append(p - lr*g)

    return tuple(updated)


In [8]:
def train(X, y, epochs=50):

    params = init_params()

    for epoch in range(epochs):

        y_pred, cache = forward(X, params)

        loss = compute_loss(y_pred, y)

        grads = backward(X, y, params, cache)

        params = update(params, grads)

        if epoch % 5 == 0:
            print(f"Epoch {epoch} | Loss: {loss:.4f}")

    return params


In [9]:
def predict(X, params):

    probs,_ = forward(X, params)

    return np.argmax(probs, axis=1)


In [10]:
pip install emnist


Note: you may need to restart the kernel to use updated packages.


In [15]:
# Load dataset
data = pd.read_csv('/kaggle/input/datasets/sachinpatel21/az-handwritten-alphabets-in-csv-format/A_Z Handwritten Data.csv')

# Split features and labels
X = data.iloc[:, 1:].values   # pixel values
y = data.iloc[:, 0].values    # labels (0–25)

# Normalize
X = X / 255.0


In [17]:
print(X.shape)  # (num_samples, 784)
print(y.shape)  # (num_samples,)


(372450, 784)
(372450,)


In [18]:
params = train(
    X[:20000],
    y[:20000],
    epochs=50
)


Epoch 0 | Loss: 3.5441
Epoch 5 | Loss: 3.4534
Epoch 10 | Loss: 3.3656
Epoch 15 | Loss: 3.2797
Epoch 20 | Loss: 3.1951
Epoch 25 | Loss: 3.1112
Epoch 30 | Loss: 3.0278
Epoch 35 | Loss: 2.9442
Epoch 40 | Loss: 2.8603
Epoch 45 | Loss: 2.7759


In [20]:
preds = predict(X[20000:20010], params)

print("Predicted:", preds)
print("Actual:   ", y[20000:20010])


Predicted: [19  0 15 19 19 19 15 19  0  1]
Actual:    [1 1 1 1 1 1 1 1 1 1]


In [21]:
def accuracy(X, y, params):
    preds = predict(X, params)
    return np.mean(preds == y)


In [22]:
# Split
X_train, X_test = X[:20000], X[20000:25000]
y_train, y_test = y[:20000], y[20000:25000]

# Train
params = train(X_train, y_train, epochs=50)

# Evaluate
acc = accuracy(X_test, y_test, params)
print("Test Accuracy:", acc * 100)


Epoch 0 | Loss: 2.9992
Epoch 5 | Loss: 2.8933
Epoch 10 | Loss: 2.7863
Epoch 15 | Loss: 2.6782
Epoch 20 | Loss: 2.5693
Epoch 25 | Loss: 2.4600
Epoch 30 | Loss: 2.3507
Epoch 35 | Loss: 2.2421
Epoch 40 | Loss: 2.1349
Epoch 45 | Loss: 2.0300
Test Accuracy: 0.3
